<a href="https://colab.research.google.com/github/CristianoMSilva/DistilBERT-FL-Playground/blob/main/04_federated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check the GPU of this session (recorded with the results)
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.6


In [2]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = load_dataset("zefang-liu/phishing-email-dataset")["train"].to_pandas()

# Clean (same steps as 01_eda.ipynb)
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Email Text"])
conflicts = df.groupby("Email Text")["Email Type"].nunique()
df = df[~df["Email Text"].isin(conflicts[conflicts > 1].index)]
df = df.drop_duplicates(subset="Email Text")
word_count = df["Email Text"].str.split().str.len()
df = df[(word_count > 0) & (word_count < 1_000_000)].reset_index(drop=True)

# Labels: 1 = phishing, 0 = safe
df["label"] = (df["Email Type"] == "Phishing Email").astype(int)

# Stratified split: 80% train, 10% validation, 10% test (same as 03_baseline_training.ipynb)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} emails, {part['label'].mean():.1%} phishing")

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Train: 14027 emails, 37.4% phishing
Validation: 1753 emails, 37.4% phishing
Test: 1754 emails, 37.4% phishing


In [3]:
from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 3

# IID split: each client gets a random third of the training set, with the same phishing ratio
skf = StratifiedKFold(n_splits=NUM_CLIENTS, shuffle=True, random_state=42)
client_dfs = [train_df.iloc[idx] for _, idx in skf.split(train_df, train_df["label"])]

for i, part in enumerate(client_dfs):
    print(f"Client {i}: {len(part)} emails, {part['label'].mean():.1%} phishing")

Client 0: 4676 emails, 37.4% phishing
Client 1: 4676 emails, 37.4% phishing
Client 2: 4675 emails, 37.4% phishing


In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

# Same settings as 03_baseline_training.ipynb (256 tokens)
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# DataFrame -> tokenized Hugging Face Dataset
def prepare(part):
    ds = Dataset.from_pandas(part[["Email Text", "label"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["Email Text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["Email Text"])

# One tokenized dataset per client; validation and test stay with the server
client_ds = [prepare(part) for part in client_dfs]
val_ds = prepare(val_df)
test_ds = prepare(test_df)

for i, ds in enumerate(client_ds):
    print(f"Client {i}: {len(ds)} emails")
print(f"Server: {len(val_ds)} validation, {len(test_ds)} test")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4676 [00:00<?, ? examples/s]

Map:   0%|          | 0/4676 [00:00<?, ? examples/s]

Map:   0%|          | 0/4675 [00:00<?, ? examples/s]

Map:   0%|          | 0/1753 [00:00<?, ? examples/s]

Map:   0%|          | 0/1754 [00:00<?, ? examples/s]

Client 0: 4676 emails
Client 1: 4676 emails
Client 2: 4675 emails
Server: 1753 validation, 1754 test


In [5]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
collator = DataCollatorWithPadding(tokenizer)

# Fresh DistilBERT with a 2-class head (same as 03)
def get_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Local training on one client: class weights come only from that client's own labels
def train_local(model, ds, epochs=1, lr=2e-5):
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(ds["label"]))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float, device=DEVICE))
    loader = DataLoader(ds, batch_size=16, shuffle=True, collate_fn=collator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.to(DEVICE).train()
    for _ in range(epochs):
        for batch in loader:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=batch["input_ids"].to(DEVICE),
                               attention_mask=batch["attention_mask"].to(DEVICE)).logits
            loss = loss_fn(logits.float(), batch["labels"].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

# Metrics with phishing (label 1) as the positive class
@torch.no_grad()
def evaluate(model, ds):
    loader = DataLoader(ds, batch_size=32, collate_fn=collator)
    model.to(DEVICE).eval()
    preds, labels = [], []
    for batch in loader:
        logits = model(input_ids=batch["input_ids"].to(DEVICE),
                       attention_mask=batch["attention_mask"].to(DEVICE)).logits
        preds += logits.argmax(-1).tolist()
        labels += batch["labels"].tolist()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall,
            "f1": f1, "false_positive_rate": fp / (fp + tn)}

# Quick test: 300 emails from client 0, evaluated on 300 validation emails
model = get_model()
start = time.time()
train_local(model, client_ds[0].select(range(300)))
print(f"Training time: {time.time() - start:.0f} s")
print({k: round(v, 4) for k, v in evaluate(model, val_ds.select(range(300))).items()})

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training time: 4 s
{'accuracy': 0.8867, 'precision': 0.7778, 'recall': 0.9633, 'f1': 0.8607, 'false_positive_rate': np.float64(0.1571)}


In [7]:
import copy
from transformers.utils import logging as hf_logging

NUM_ROUNDS = 5

# Fresh model without the UNEXPECTED/MISSING report (it would print every round)
def get_model():
    hf_logging.set_verbosity_error()
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

global_model = get_model()
sizes = [len(ds) for ds in client_ds]
history, best = [], {"f1": -1.0}
start = time.time()

for rnd in range(1, NUM_ROUNDS + 1):
    # 1) Each client starts from the global weights and trains 1 epoch on its own emails
    client_states = []
    for ds in client_ds:
        local_model = copy.deepcopy(global_model)
        train_local(local_model, ds)
        client_states.append({k: v.detach().cpu() for k, v in local_model.state_dict().items()})
        del local_model

    # 2) Server: FedAvg = average of the client weights, weighted by number of emails
    avg_state = {}
    for k, v in client_states[0].items():
        if v.is_floating_point():
            avg_state[k] = sum(s[k] * n for s, n in zip(client_states, sizes)) / sum(sizes)
        else:
            avg_state[k] = v
    global_model.load_state_dict(avg_state)

    # 3) Server evaluates the new global model on the validation set and keeps the best round
    metrics = evaluate(global_model, val_ds)
    history.append({"round": rnd, **metrics})
    print(f"Round {rnd} ({(time.time() - start) / 60:.1f} min): "
          + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    if metrics["f1"] > best["f1"]:
        best.update(f1=metrics["f1"], round=rnd, state=copy.deepcopy(global_model.state_dict()))

fl_time = time.time() - start
print(f"Federated training time: {fl_time / 60:.1f} min | best round: {best['round']} (F1 {best['f1']:.4f})")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Round 1 (1.8 min): accuracy=0.9846, precision=0.9687, recall=0.9908, f1=0.9796, false_positive_rate=0.0191
Round 2 (3.6 min): accuracy=0.9886, precision=0.9833, recall=0.9863, f1=0.9848, false_positive_rate=0.0100
Round 3 (5.3 min): accuracy=0.9909, precision=0.9848, recall=0.9908, f1=0.9878, false_positive_rate=0.0091
Round 4 (7.1 min): accuracy=0.9909, precision=0.9893, recall=0.9863, f1=0.9878, false_positive_rate=0.0064
Round 5 (8.9 min): accuracy=0.9909, precision=0.9908, recall=0.9847, f1=0.9877, false_positive_rate=0.0055
Federated training time: 8.9 min | best round: 3 (F1 0.9878)


In [8]:
# Final test: best round's global model on the test set (same 1,754 emails as 03)
global_model.load_state_dict(best["state"])
fl_test = evaluate(global_model, test_ds)
fl_test["train_time_min"] = fl_time / 60

# Comparison with the centralized baseline (256 tokens, numbers from 03_baseline_training.ipynb)
results = {
    "Centralized (256)": {"accuracy": 0.9863, "precision": 0.9847, "recall": 0.9787, "f1": 0.9817,
                          "false_positive_rate": 0.0091, "train_time_min": 6.0},
    "Federated IID, 3 clients (256)": fl_test,
}
pd.DataFrame(results).T.astype(float).round(4)

,accuracy,precision,recall,f1,false_positive_rate,train_time_min
Centralized (256),0.9863,0.9847,0.9787,0.9817,0.0091,6.0000
"Federated IID, 3 clients (256)",0.9869,0.9759,0.9893,0.9826,0.0146,8.8683


## Results: federated (IID) vs centralized

Both use DistilBERT with `max_length` 256 and the same test set (1,754 emails). The federated run uses manual FedAvg: 3 IID clients, 5 rounds, 1 local epoch per round, best round chosen by validation F1 (round 3).

- **F1 was practically the same**: 98.26% federated vs 98.17% centralized. Training without pooling the data cost almost nothing in this IID setting.
- **The type of error changed**: federated missed fewer phishing emails (7 vs 14) but raised more false alarms (16 vs 10).
- **Training time is not comparable**: the 3 clients trained one after another on the same GPU (in practice they would train in parallel), and 5 rounds equal 5 epochs vs 3 in the centralized run.
- **Caveat**: single run (seed 42); differences of a few emails show a trend, not proof.

*Note: Flower 1.39 (`run_simulation` with Ray) crashed on the Colab GPU, so FedAvg was implemented manually with the same algorithm.*

In [9]:
# Non-IID split: phishing and safe emails are spread unevenly across the 3 clients
PHISH_SHARE = [0.6, 0.3, 0.1]   # share of all phishing emails each client gets
SAFE_SHARE = [0.1, 0.3, 0.6]    # share of all safe emails each client gets

def uneven_split(part, shares):
    part = part.sample(frac=1, random_state=42)
    cuts = [0, *(np.cumsum(shares)[:-1] * len(part)).astype(int), len(part)]
    return [part.iloc[a:b] for a, b in zip(cuts[:-1], cuts[1:])]

phish_parts = uneven_split(train_df[train_df["label"] == 1], PHISH_SHARE)
safe_parts = uneven_split(train_df[train_df["label"] == 0], SAFE_SHARE)
noniid_dfs = [pd.concat([p, s]) for p, s in zip(phish_parts, safe_parts)]

for i, part in enumerate(noniid_dfs):
    print(f"Client {i}: {len(part)} emails, {part['label'].mean():.1%} phishing")

# Tokenize each client's emails (same function as the IID clients)
noniid_ds = [prepare(part) for part in noniid_dfs]

Client 0: 4025 emails, 78.2% phishing
Client 1: 4207 emails, 37.4% phishing
Client 2: 5795 emails, 9.1% phishing


Map:   0%|          | 0/4025 [00:00<?, ? examples/s]

Map:   0%|          | 0/4207 [00:00<?, ? examples/s]

Map:   0%|          | 0/5795 [00:00<?, ? examples/s]

In [12]:
# Same FedAvg loop as cell 6, as a function so it can run on any set of clients
def run_fedavg(clients, num_rounds=NUM_ROUNDS):
    global_model = get_model()
    sizes = [len(ds) for ds in clients]
    history, best = [], {"f1": -1.0}
    start = time.time()
    for rnd in range(1, num_rounds + 1):
        # 1) Local training on each client
        client_states = []
        for ds in clients:
            local_model = copy.deepcopy(global_model)
            train_local(local_model, ds)
            client_states.append({k: v.detach().cpu() for k, v in local_model.state_dict().items()})
            del local_model
        # 2) FedAvg: weighted average by number of emails
        avg_state = {}
        for k, v in client_states[0].items():
            if v.is_floating_point():
                avg_state[k] = sum(s[k] * n for s, n in zip(client_states, sizes)) / sum(sizes)
            else:
                avg_state[k] = v
        global_model.load_state_dict(avg_state)
        # 3) Validation and best round
        metrics = evaluate(global_model, val_ds)
        history.append({"round": rnd, **metrics})
        print(f"Round {rnd} ({(time.time() - start) / 60:.1f} min): "
              + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
        if metrics["f1"] > best["f1"]:
            best.update(f1=metrics["f1"], round=rnd, state=copy.deepcopy(global_model.state_dict()))
    best["time_min"] = (time.time() - start) / 60
    print(f"Training time: {best['time_min']:.1f} min | best round: {best['round']} (F1 {best['f1']:.4f})")
    global_model.load_state_dict(best["state"])
    return global_model, history

# Run on the non-IID clients and test the best round
start = time.time()
noniid_model, noniid_history = run_fedavg(noniid_ds)
noniid_test = evaluate(noniid_model, test_ds)
noniid_test["train_time_min"] = (time.time() - start) / 60

# Add to the comparison table
results["Federated non-IID, 3 clients (256)"] = noniid_test
pd.DataFrame(results).T.astype(float).round(4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Round 1 (1.8 min): accuracy=0.9783, precision=0.9753, recall=0.9664, f1=0.9709, false_positive_rate=0.0146
Round 2 (3.5 min): accuracy=0.9840, precision=0.9658, recall=0.9924, f1=0.9789, false_positive_rate=0.0209
Round 3 (5.3 min): accuracy=0.9875, precision=0.9759, recall=0.9908, f1=0.9833, false_positive_rate=0.0146
Round 4 (7.0 min): accuracy=0.9875, precision=0.9803, recall=0.9863, f1=0.9833, false_positive_rate=0.0118
Round 5 (8.7 min): accuracy=0.9886, precision=0.9892, recall=0.9802, f1=0.9847, false_positive_rate=0.0064
Training time: 8.7 min | best round: 5 (F1 0.9847)


,accuracy,precision,recall,f1,false_positive_rate,train_time_min
Centralized (256),0.9863,0.9847,0.9787,0.9817,0.0091,6.0000
"Federated IID, 3 clients (256)",0.9869,0.9759,0.9893,0.9826,0.0146,8.8683
"Federated non-IID, 3 clients (256)",0.9880,0.9862,0.9817,0.9840,0.0082,8.9435
